# Deep Learning 003 — A Tour of the Architectures

Four architectures exist because four kinds of structure exist. The useful exercise is
not running each on data that suits it — it is running each on data that **doesn't**,
because the failure is what tells you what the architecture assumed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. Tabular data — rows of independent columns. An ANN is the right tool.

There is no spatial or sequential structure in a customer record. Column order is
arbitrary, and a dense network treats every column the same way — which is exactly
correct here.

In [ ]:
ch = pd.read_csv('../data/Churn_Modelling.csv')
num = ch[['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
          'EstimatedSalary']].to_numpy()
yc = ch['Exited'].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(num, yc, test_size=0.2,
                                      random_state=0, stratify=yc)
sc = StandardScaler().fit(Xtr)
ann = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=600,
                    random_state=0).fit(sc.transform(Xtr), ytr)
print(f'ANN on tabular churn   test accuracy {ann.score(sc.transform(Xte), yte):.1%}')
print(f'majority baseline      {max(np.bincount(yte))/len(yte):.1%}')

### Now the diagnostic: shuffle the column order.

If the architecture assumed nothing about column order, shuffling should change
nothing. Watch.

In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(num.shape[1])
ann_p = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=600,
                      random_state=0).fit(sc.transform(Xtr)[:, perm], ytr)
print(f'columns shuffled       test accuracy '
      f'{ann_p.score(sc.transform(Xte)[:, perm], yte):.1%}')
print('\nEssentially unchanged. A dense layer is PERMUTATION INVARIANT over its')
print('inputs - it has no notion of which input sits next to which. For tabular')
print('data that is the right assumption. Keep that sentence; the next section')
print('turns it into a problem.')

## 2. Images — a grid, where *neighbouring* pixels mean something

Run the same dense network on 8×8 digit images. It works. Then shuffle the pixels — the
**same** shuffle for every image, so no information is destroyed, only the spatial
arrangement.

In [ ]:
d = load_digits()
Xi, yi = d.data, d.target
Itr, Ite, jtr, jte = train_test_split(Xi, yi, test_size=0.25,
                                      random_state=0, stratify=yi)

mlp = MLPClassifier(hidden_layer_sizes=(32,), max_iter=800,
                    random_state=0).fit(Itr / 16, jtr)
print(f'dense net on digits             {mlp.score(Ite / 16, jte):.1%}')

pix = rng.permutation(64)
mlp_s = MLPClassifier(hidden_layer_sizes=(32,), max_iter=800,
                      random_state=0).fit(Itr[:, pix] / 16, jtr)
print(f'dense net on SHUFFLED pixels    {mlp_s.score(Ite[:, pix] / 16, jte):.1%}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(Xi[0].reshape(8, 8), cmap='gray_r'); ax[0].set_title('original')
ax[1].imshow(Xi[0][pix].reshape(8, 8), cmap='gray_r'); ax[1].set_title('pixels shuffled')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

**The two accuracies are nearly identical — and that is the problem.**

You cannot read the shuffled digit. The dense network does not care, because it never
used the fact that pixel 9 sits below pixel 1. **It is throwing away the grid.** A CNN
is the architecture that refuses to: its kernel only ever looks at neighbours, so
shuffling the pixels would wreck it.

That is what "a CNN has an inductive bias about locality" means, and this cell is what
it looks like when the bias is missing.

## 3. Sequences — where position carries the meaning

A dense network sees a fixed-size vector. Give it a task where the *same* symbols in a
different order mean different things.

In [ ]:
# Task: does the sequence contain the pattern [1, 2] in that order?
def make(n, length=8, seed=0):
    r = np.random.default_rng(seed)
    S = r.integers(0, 3, size=(n, length))
    lab = np.array([int(any(s[i] == 1 and s[i+1] == 2 for i in range(length - 1)))
                    for s in S])
    return S, lab

S, ls = make(4000, seed=1)
Str, Ste, ltr, lte = train_test_split(S, ls, test_size=0.25, random_state=0)
seq = MLPClassifier(hidden_layer_sizes=(32,), max_iter=1500,
                    random_state=0).fit(Str, ltr)
print(f'dense net, order matters       {seq.score(Ste, lte):.1%}')
print(f'majority baseline              {max(np.bincount(lte))/len(lte):.1%}')

# The bag-of-symbols version: same counts, order destroyed.
bag_tr = np.stack([np.bincount(s, minlength=3) for s in Str])
bag_te = np.stack([np.bincount(s, minlength=3) for s in Ste])
bag = MLPClassifier(hidden_layer_sizes=(32,), max_iter=1500,
                    random_state=0).fit(bag_tr, ltr)
print(f'same net on symbol COUNTS only {bag.score(bag_te, lte):.1%}')

**The dense net solves this one outright**, and the counts-only control is why that is
still informative: strip the order away and accuracy falls a long way, so the label
really does depend on position and the network really is using it.

So where is the problem? In *how* it uses it. The network must learn "1 at position 3
followed by 2 at position 4" **separately** from the same pattern at positions 5 and 6 —
it gets no help from the fact that they are the same pattern. With 8 positions and 4,000
examples that is affordable. An RNN or a transformer shares that machinery across all
positions, so it never pays the cost at all, which is why the gap opens as sequences
grow longer and data gets scarcer. Exercise 2 is where you make it appear.

## The summary the lesson is making

| Data | Structure to exploit | Architecture | What breaks without it |
|---|---|---|---|
| Tabular | none — columns are independent | ANN | nothing; ANN is correct here |
| Images | locality on a grid | CNN | a dense net is indifferent to shuffled pixels |
| Sequences | order and reuse across positions | RNN / transformer | patterns must be relearned at every position |

## Exercises

1. Re-run section 2 with a `(256, 128)` dense net. Does more capacity recover the
   spatial information, or just memorise?
2. In section 3, raise `length` from 8 to 24. Which model degrades faster?
3. Shuffle the pixels **differently for each image** and re-run. Explain why the
   accuracy collapses now when it did not before.